# Sentinela Inline Binary Temporal GOES Pipeline

This notebook is self-contained: no GitHub clone, no Kaggle Dataset, and no calls to `scripts/*.py`.

It downloads FIRMS bootstrap labels, streams GOES temporal windows, trains Sentinela, evaluates, and writes outputs directly from notebook cells.

Run on Kaggle with GPU + Internet enabled, or on Colab with GPU + Internet enabled.

In [ ]:
# Environment setup
# Kaggle sometimes assigns Tesla P100 (compute capability sm_60).
# Newer PyTorch CUDA wheels can omit sm_60 kernels, causing:
#   CUDA error: no kernel image is available for execution on the device
# Install a CUDA 11.8 PyTorch build first, then the geospatial/data deps.
%pip uninstall -y torch torchvision torchaudio -q
%pip install -q --no-cache-dir torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu118
%pip install -q numpy pyyaml tqdm rasterio matplotlib pillow scikit-learn xarray netCDF4 boto3 botocore pyproj

import torch
print('torch', torch.__version__, 'torch_cuda', torch.version.cuda)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0), 'capability', torch.cuda.get_device_capability(0))
    print('compiled_arches', torch.cuda.get_arch_list())


In [ ]:
# Parameters
from pathlib import Path
import os

IS_KAGGLE = Path('/kaggle/working').exists()
WORK_ROOT = Path('/kaggle/working' if IS_KAGGLE else '/content/sentinela_inline_work')
DATA_ROOT = WORK_ROOT / 'goes_fire'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

REGION = 'south_america'
START_DATE = '2024-01-01'
END_DATE = '2026-05-11'
MIN_CONFIDENCE = 'nominal'
MAX_FIRMS_ROWS = 2000
FIRMS_MAP_KEY = ''  # Prefer Kaggle Secret named FIRMS_MAP_KEY. Paste only if needed.

TEMPORAL_OFFSETS = '-30,-20,-10,0'
SAMPLES_PER_CLASS = 1000
MAX_GOES_FILES = 8       # Use 5-10 for smoke, 0 for full selected targets.
MAX_TARGETS_PER_FILE = 0
HARD_NEGATIVE_RATIO = 0.5
KEEP_RAW = False
SEED = 7

VARIANT = 'n'            # n for smoke, s/m for real training.
EPOCHS = 1               # 1 for smoke, 20 for real training.
BATCH_SIZE = 16
NUM_WORKERS = 2
LR = 3e-4
WEIGHT_DECAY = 0.01
THRESHOLD = 0.5

print({'is_kaggle': IS_KAGGLE, 'work_root': str(WORK_ROOT), 'data_root': str(DATA_ROOT)})

In [ ]:
# Regional config and constants
from dataclasses import dataclass
from datetime import date, datetime, time, timedelta, timezone
from pathlib import Path
import csv, hashlib, io, json, os, random, re, urllib.request, urllib.parse, urllib.error
from collections import Counter, defaultdict
from typing import Any

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

RAW_GOES_CHANNELS = [
    'CMI_C02', 'CMI_C03', 'CMI_C05', 'CMI_C06', 'CMI_C07',
    'CMI_C11', 'CMI_C13', 'CMI_C14', 'CMI_C15',
]
LABEL_NAMES = {0: 'negative', 1: 'active_fire', 2: 'early_fire_signal', 3: 'hard_negative', 4: 'uncertain'}
BINARY_LABEL_NAMES = {0: 'no_fire', 1: 'fire_signal'}
FIRE_SIGNAL_LABELS = {1, 2}
NO_FIRE_LABELS = {0, 3}
UNCERTAIN_LABELS = {4}
HARD_NEGATIVE_TYPES = ['ocean_glint', 'coastline', 'cloud_edge', 'deep_convective_cloud', 'thin_cirrus', 'desert_hot_surface', 'urban_industrial_heat', 'volcano_geothermal', 'dust', 'fog_low_cloud', 'sensor_edge_artifact', 'agricultural_burn', 'unknown_hotspot']

REGIONS = {
    'south_america': {'bbox': (-82.0, -56.0, -34.0, 13.0), 'source_id': 'goes_east_abi', 'bucket': 'noaa-goes19', 'fallback_buckets': ('noaa-goes16',), 'product': 'ABI-L2-MCMIPF', 'sector': 'full_disk'},
    'central_america': {'bbox': (-118.0, 5.0, -76.0, 24.0), 'source_id': 'goes_east_abi', 'bucket': 'noaa-goes19', 'fallback_buckets': ('noaa-goes16',), 'product': 'ABI-L2-MCMIPF', 'sector': 'full_disk'},
    'caribbean': {'bbox': (-90.0, 7.0, -58.0, 28.0), 'source_id': 'goes_east_abi', 'bucket': 'noaa-goes19', 'fallback_buckets': ('noaa-goes16',), 'product': 'ABI-L2-MCMIPF', 'sector': 'full_disk'},
    'north_america': {'bbox': (-168.0, 12.0, -52.0, 72.0), 'source_id': 'goes_east_west_abi', 'bucket': 'noaa-goes19', 'fallback_buckets': ('noaa-goes18', 'noaa-goes16', 'noaa-goes17'), 'product': 'ABI-L2-MCMIPF', 'sector': 'full_disk'},
}

@dataclass(frozen=True)
class RegionalConfig:
    name: str
    bbox: tuple[float, float, float, float]
    goes_source: str
    bucket: str
    fallback_buckets: tuple[str, ...]
    product: str = 'ABI-L2-MCMIPF'
    sector: str = 'full_disk'
    raw_channels: tuple[str, ...] = tuple(RAW_GOES_CHANNELS)
    patch_size: int = 64
    temporal_tolerance_minutes: int = 15
    early_signal_minutes: int = 60

    def region_root(self, data_root: str | Path) -> Path:
        return Path(data_root) / self.name


def get_region_config(region: str) -> RegionalConfig:
    row = REGIONS[region]
    return RegionalConfig(name=region, bbox=row['bbox'], goes_source=row['source_id'], bucket=row['bucket'], fallback_buckets=tuple(row['fallback_buckets']), product=row['product'], sector=row['sector'])


def parse_utc(value: str) -> datetime:
    text = str(value).strip()
    if text.endswith('Z'):
        text = text[:-1] + '+00:00'
    dt = datetime.fromisoformat(text)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def format_utc(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).isoformat().replace('+00:00', 'Z')


def parse_temporal_offsets(text: str) -> tuple[int, ...]:
    return tuple(int(v.strip()) for v in str(text).split(',') if v.strip())


def split_for_id(sample_id: str) -> str:
    n = int(hashlib.sha1(sample_id.encode('utf-8')).hexdigest()[:8], 16) % 100
    return 'train' if n < 80 else 'val' if n < 90 else 'test'


def folder_for_label(label: int) -> str:
    return {0: 'negative', 1: 'positive', 2: 'early_positive', 3: 'hard_negative', 4: 'uncertain'}[int(label)]


def binary_label_for(raw_label: int) -> int:
    if raw_label in FIRE_SIGNAL_LABELS:
        return 1
    if raw_label in NO_FIRE_LABELS or raw_label in UNCERTAIN_LABELS:
        return 0
    raise ValueError(raw_label)

In [ ]:
# Model: Sentinela body with temporal flattening adapter
from dataclasses import dataclass
import math
from typing import Any

import torch
import torch.nn as nn
import torch.nn.functional as F


def make_divisible(value: float, divisor: int = 8) -> int:
    return max(divisor, int(math.ceil(float(value) / float(divisor)) * divisor))


def scale_channels(base: int, width_mult: float, max_channels: int, divisor: int = 8) -> int:
    ch = make_divisible(int(base * float(width_mult)), divisor=divisor)
    return int(min(ch, int(max_channels)))


def scale_depth(base_repeats: int, depth_mult: float) -> int:
    return max(1, int(round(float(base_repeats) * float(depth_mult))))


class ConvBNAct(nn.Module):
    def __init__(self, c_in: int, c_out: int, k: int = 1, s: int = 1, p: int | None = None, g: int = 1):
        super().__init__()
        pad = (int(k) - 1) // 2 if p is None else int(p)
        self.conv = nn.Conv2d(c_in, c_out, kernel_size=int(k), stride=int(s), padding=pad, groups=int(g), bias=False)
        self.bn = nn.BatchNorm2d(c_out)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.bn(self.conv(x)))


class Bottleneck(nn.Module):
    def __init__(self, channels: int, shortcut: bool = True, e: float = 1.0, k: int = 3):
        super().__init__()
        hidden = max(8, int(round(int(channels) * float(e))))
        self.cv1 = ConvBNAct(channels, hidden, k=1, s=1)
        self.cv2 = ConvBNAct(hidden, channels, k=int(k), s=1)
        self.use_shortcut = bool(shortcut)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.cv2(self.cv1(x))
        return y + x if self.use_shortcut and y.shape == x.shape else y


class C3k2(nn.Module):
    def __init__(self, c_in: int, c_out: int, n: int = 2, c3k: bool = False, e: float = 0.5):
        super().__init__()
        hidden = max(8, int(round(int(c_out) * float(e))))
        self.cv1 = ConvBNAct(c_in, 2 * hidden, k=1, s=1)
        k = 5 if bool(c3k) else 3
        self.blocks = nn.ModuleList([Bottleneck(hidden, shortcut=True, e=1.0, k=k) for _ in range(int(n))])
        self.cv2 = ConvBNAct((2 + int(n)) * hidden, c_out, k=1, s=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.cv1(x)
        a, b = y.chunk(2, dim=1)
        feats = [a, b]
        h = b
        for block in self.blocks:
            h = block(h)
            feats.append(h)
        return self.cv2(torch.cat(feats, dim=1))


class SPPF(nn.Module):
    def __init__(self, c_in: int, c_out: int, k: int = 5):
        super().__init__()
        hidden = max(8, int(c_in) // 2)
        self.cv1 = ConvBNAct(c_in, hidden, k=1, s=1)
        self.pool = nn.MaxPool2d(kernel_size=int(k), stride=1, padding=int(k) // 2)
        self.cv2 = ConvBNAct(hidden * 4, c_out, k=1, s=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.cv1(x)
        y1 = self.pool(x)
        y2 = self.pool(y1)
        y3 = self.pool(y2)
        return self.cv2(torch.cat([x, y1, y2, y3], dim=1))


class ChannelSE(nn.Module):
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        hidden = max(8, int(channels) // int(reduction))
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, kernel_size=1),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.net(x)


class UpBlock(nn.Module):
    def __init__(self, c_in: int, c_skip: int, c_out: int, repeats: int = 2):
        super().__init__()
        self.reduce = ConvBNAct(c_in, c_out, k=1, s=1)
        self.fuse = C3k2(c_out + c_skip, c_out, n=int(repeats), c3k=True, e=0.5)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        x = self.reduce(x)
        return self.fuse(torch.cat([x, skip], dim=1))


@dataclass(frozen=True)
class SentinelaConfig:
    in_channels: int = 36
    mask_classes: int = 1
    scene_classes: int = 1
    temporal_steps: int = 4
    input_channels_per_timestep: int | None = 9
    variant: str = 's'
    input_size: int = 64
    base_channels: int = 32
    width_mult: float = 1.0
    depth_mult: float = 1.0
    max_channels: int = 384
    dropout: float = 0.05
    include_scene_head: bool = True


VARIANTS = {
    'n': {'base_channels': 24, 'width_mult': 0.75, 'depth_mult': 0.67, 'max_channels': 256},
    's': {'base_channels': 32, 'width_mult': 1.0, 'depth_mult': 1.0, 'max_channels': 384},
    'm': {'base_channels': 48, 'width_mult': 1.15, 'depth_mult': 1.35, 'max_channels': 512},
    'l': {'base_channels': 64, 'width_mult': 1.25, 'depth_mult': 1.5, 'max_channels': 1024},
}


class SpectralInputAdapter(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.in_channels = int(in_channels)
        self.proj = ConvBNAct(self.in_channels, int(out_channels), k=3, s=1)
        self.mix = nn.Sequential(ConvBNAct(int(out_channels), int(out_channels), k=1, s=1), ChannelSE(int(out_channels)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 4:
            raise ValueError(f'Expected [B,C,H,W], got {tuple(x.shape)}')
        if x.shape[1] != self.in_channels:
            raise ValueError(f'Expected {self.in_channels} input bands, got {x.shape[1]}')
        return self.mix(self.proj(x))


class TemporalInputAdapter(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        self.in_channels = int(in_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim == 5:
            b, t, c, h, w = x.shape
            x = x.reshape(b, t * c, h, w)
        elif x.ndim != 4:
            raise ValueError(f'Expected [B,C,H,W] or [B,T,C,H,W], got {tuple(x.shape)}')
        if x.shape[1] != self.in_channels:
            raise ValueError(f'Expected {self.in_channels} flattened bands, got {x.shape[1]}')
        return x


class SentinelaEncoder(nn.Module):
    def __init__(self, cfg: SentinelaConfig):
        super().__init__()
        c1 = scale_channels(cfg.base_channels, cfg.width_mult, cfg.max_channels)
        c2 = scale_channels(cfg.base_channels * 2, cfg.width_mult, cfg.max_channels)
        c3 = scale_channels(cfg.base_channels * 4, cfg.width_mult, cfg.max_channels)
        c4 = scale_channels(cfg.base_channels * 8, cfg.width_mult, cfg.max_channels)
        c5 = scale_channels(cfg.base_channels * 10, cfg.width_mult, cfg.max_channels)
        d1, d2, d3 = scale_depth(1, cfg.depth_mult), scale_depth(2, cfg.depth_mult), scale_depth(2, cfg.depth_mult)
        self.channels = [c1, c2, c3, c4, c5]
        self.adapter = SpectralInputAdapter(cfg.in_channels, c1)
        self.stem = ConvBNAct(c1, c1, k=3, s=1)
        self.down1 = ConvBNAct(c1, c2, k=3, s=2)
        self.stage1 = C3k2(c2, c2, n=d1, c3k=False)
        self.down2 = ConvBNAct(c2, c3, k=3, s=2)
        self.stage2 = C3k2(c3, c3, n=d2, c3k=False)
        self.down3 = ConvBNAct(c3, c4, k=3, s=2)
        self.stage3 = C3k2(c4, c4, n=d3, c3k=True)
        self.down4 = ConvBNAct(c4, c5, k=3, s=2)
        self.stage4 = nn.Sequential(C3k2(c5, c5, n=d2, c3k=True), SPPF(c5, c5))

    def forward(self, x: torch.Tensor) -> list[torch.Tensor]:
        p1 = self.stem(self.adapter(x))
        p2 = self.stage1(self.down1(p1))
        p3 = self.stage2(self.down2(p2))
        p4 = self.stage3(self.down3(p3))
        p5 = self.stage4(self.down4(p4))
        return [p1, p2, p3, p4, p5]


class SegmentationHead(nn.Module):
    def __init__(self, channels: list[int], mask_classes: int, dropout: float = 0.05):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.up4 = UpBlock(c5, c4, c4)
        self.up3 = UpBlock(c4, c3, c3)
        self.up2 = UpBlock(c3, c2, c2)
        self.up1 = UpBlock(c2, c1, c1)
        self.refine = nn.Sequential(ConvBNAct(c1, c1, k=3, s=1), nn.Dropout2d(float(dropout)), ConvBNAct(c1, c1, k=3, s=1))
        self.mask_logits = nn.Conv2d(c1, int(mask_classes), kernel_size=1)

    def forward(self, feats: list[torch.Tensor], output_size: tuple[int, int]) -> torch.Tensor:
        p1, p2, p3, p4, p5 = feats
        x = self.up4(p5, p4)
        x = self.up3(x, p3)
        x = self.up2(x, p2)
        x = self.up1(x, p1)
        x = self.mask_logits(self.refine(x))
        return F.interpolate(x, size=output_size, mode='bilinear', align_corners=False) if x.shape[-2:] != output_size else x


class SceneHead(nn.Module):
    def __init__(self, channels: int, scene_classes: int = 1, dropout: float = 0.05):
        super().__init__()
        hidden = max(32, channels // 2)
        self.scene_classes = int(scene_classes)
        self.net = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(channels, hidden), nn.SiLU(inplace=True), nn.Dropout(float(dropout)), nn.Linear(hidden, self.scene_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.net(x)
        return logits.squeeze(-1) if self.scene_classes == 1 else logits


class SentinelaModel(nn.Module):
    def __init__(self, config: SentinelaConfig | None = None, **kwargs: Any):
        super().__init__()
        cfg = config or SentinelaConfig(**kwargs)
        variant = str(cfg.variant).lower().strip()
        if variant in VARIANTS:
            p = VARIANTS[variant]
            cfg = SentinelaConfig(
                in_channels=cfg.in_channels, mask_classes=cfg.mask_classes, scene_classes=cfg.scene_classes,
                temporal_steps=cfg.temporal_steps, input_channels_per_timestep=cfg.input_channels_per_timestep,
                variant=variant, input_size=cfg.input_size, base_channels=int(p['base_channels']),
                width_mult=float(p['width_mult']), depth_mult=float(p['depth_mult']), max_channels=int(p['max_channels']),
                dropout=cfg.dropout, include_scene_head=cfg.include_scene_head,
            )
        self.config = cfg
        self.temporal_adapter = TemporalInputAdapter(cfg.in_channels)
        self.encoder = SentinelaEncoder(cfg)
        self.seg_head = SegmentationHead(self.encoder.channels, cfg.mask_classes, dropout=cfg.dropout)
        self.scene_head = SceneHead(self.encoder.channels[-1], scene_classes=cfg.scene_classes, dropout=cfg.dropout) if cfg.include_scene_head else None

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        output_size = tuple(x.shape[-2:])
        x = self.temporal_adapter(x)
        feats = self.encoder(x)
        out = {'mask_logits': self.seg_head(feats, output_size=output_size)}
        if self.scene_head is not None:
            out['scene_logits'] = self.scene_head(feats[-1])
        return out

In [ ]:
# FIRMS bootstrap label download and ingest, inline
DEFAULT_FIRMS_SOURCES = 'VIIRS_SNPP_SP,VIIRS_NOAA20_SP,VIIRS_NOAA21_SP,MODIS_SP'
FIRMS_BASE_URL = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv'


def get_firms_key() -> str:
    if FIRMS_MAP_KEY:
        return FIRMS_MAP_KEY
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore
        key = UserSecretsClient().get_secret('FIRMS_MAP_KEY')
        if key:
            return key
    except Exception:
        pass
    import getpass
    return getpass.getpass('NASA FIRMS map key: ')


def normalize_confidence(value: str) -> str:
    text = str(value or '').strip().lower()
    return {'l': 'low', 'n': 'nominal', 'h': 'high'}.get(text, text)


def confidence_ok(value: str, minimum: str) -> bool:
    if not minimum:
        return True
    conf = normalize_confidence(value)
    order = {'low': 0, 'nominal': 1, 'high': 2}
    if minimum.lower() in order:
        return order.get(conf, 0) >= order[minimum.lower()]
    try:
        return float(value) >= float(minimum)
    except ValueError:
        return True


def acq_datetime_utc(row: dict[str, str]) -> str:
    acq_time = str(row['acq_time']).strip().zfill(4)
    dt = datetime.strptime(f"{row['acq_date']} {acq_time}", '%Y-%m-%d %H%M').replace(tzinfo=timezone.utc)
    return format_utc(dt)


def event_id_for(source: str, row: dict[str, str]) -> str:
    payload = '|'.join([source, row.get('latitude', ''), row.get('longitude', ''), row.get('acq_date', ''), str(row.get('acq_time', '')).zfill(4), row.get('satellite', ''), row.get('instrument', '')])
    return 'firms-' + hashlib.sha1(payload.encode('utf-8')).hexdigest()[:16]


def fetch_firms_csv(map_key: str, source: str, bbox: tuple[float, float, float, float], start: date, day_range: int) -> list[dict[str, str]]:
    bbox_text = ','.join(f'{v:g}' for v in bbox)
    url = '/'.join([FIRMS_BASE_URL, urllib.parse.quote(map_key), urllib.parse.quote(source), urllib.parse.quote(bbox_text, safe=','), str(day_range), start.isoformat()])
    with urllib.request.urlopen(url, timeout=120) as response:
        text = response.read().decode('utf-8')
    if not text.strip() or text.lstrip().startswith('Invalid'):
        return []
    return list(csv.DictReader(io.StringIO(text)))


def iter_chunks(start: date, end: date, chunk_days: int = 5) -> list[tuple[date, int]]:
    out = []
    current = start
    chunk = max(1, min(5, int(chunk_days)))
    while current <= end:
        days = min(chunk, (end - current).days + 1)
        out.append((current, days))
        current += timedelta(days=days)
    return out


def download_firms_labels(cfg: RegionalConfig, data_root: Path) -> Path:
    map_key = get_firms_key()
    start = date.fromisoformat(START_DATE)
    end = date.fromisoformat(END_DATE)
    out_path = cfg.region_root(data_root) / 'raw' / 'labels' / 'firms_bootstrap_events.csv'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sources = [s.strip() for s in DEFAULT_FIRMS_SOURCES.split(',') if s.strip()]
    labels: dict[str, dict[str, str]] = {}
    failures = []
    chunks = iter_chunks(start, end, chunk_days=5)
    for source in sources:
        for idx, (chunk_start, days) in enumerate(chunks, start=1):
            try:
                rows = fetch_firms_csv(map_key, source, cfg.bbox, chunk_start, days)
            except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as exc:
                failures.append(f'{source} {chunk_start}: {exc}')
                continue
            for row in rows:
                if not {'latitude', 'longitude', 'acq_date', 'acq_time'} <= set(row):
                    continue
                if not confidence_ok(row.get('confidence', ''), MIN_CONFIDENCE):
                    continue
                eid = event_id_for(source, row)
                labels[eid] = {
                    'event_id': eid,
                    'event_time_utc': acq_datetime_utc(row),
                    'lat': f"{float(row['latitude']):.6f}",
                    'lon': f"{float(row['longitude']):.6f}",
                    'label': '1',
                    'label_name': 'active_fire',
                    'source': f'firms_bootstrap:{source}',
                    'confidence': normalize_confidence(row.get('confidence', '')),
                }
                if MAX_FIRMS_ROWS > 0 and len(labels) >= MAX_FIRMS_ROWS:
                    break
            if MAX_FIRMS_ROWS > 0 and len(labels) >= MAX_FIRMS_ROWS:
                break
            if idx % 20 == 0 or idx == len(chunks):
                print(f'source={source} chunks={idx}/{len(chunks)} labels={len(labels)}', flush=True)
        if MAX_FIRMS_ROWS > 0 and len(labels) >= MAX_FIRMS_ROWS:
            break
    rows_out = sorted(labels.values(), key=lambda r: (r['event_time_utc'], r['event_id']))
    with out_path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=['event_id', 'event_time_utc', 'lat', 'lon', 'label', 'label_name', 'source', 'confidence'])
        writer.writeheader(); writer.writerows(rows_out)
    print(json.dumps({'labels': len(rows_out), 'out': str(out_path), 'failures': failures[:5], 'failure_count': len(failures)}, indent=2))
    return out_path


def ingest_labels(raw_path: Path, cfg: RegionalConfig, data_root: Path) -> Path:
    out_path = cfg.region_root(data_root) / 'processed' / 'labels.csv'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with raw_path.open(newline='', encoding='utf-8') as handle:
        rows = list(csv.DictReader(handle))
    with out_path.open('w', newline='', encoding='utf-8') as handle:
        fieldnames = ['event_id', 'event_time_utc', 'lat', 'lon', 'label', 'label_name', 'source', 'confidence', 'notes']
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({**{k: row.get(k, '') for k in fieldnames}, 'notes': row.get('notes', '')})
    print(json.dumps({'rows': len(rows), 'out': str(out_path)}, indent=2))
    return out_path

In [ ]:
# Streaming GOES temporal sample builder, inline
GOES_START_RE = re.compile(r'_s(?P<stamp>\d{14})')
MANIFEST_COLUMNS = ['sample_id','region','source_id','sector','timestamp_utc','center_lat','center_lon','bbox_w','bbox_s','bbox_e','bbox_n','patch_row0','patch_col0','patch_size','label','label_name','label_source','event_id','event_time_utc','delta_t_minutes','hard_negative_type','split','path']


def read_csv(path: Path) -> list[dict[str, str]]:
    if not path.exists():
        return []
    with path.open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))


def append_csv(path: Path, rows: list[dict[str, Any]], fieldnames: list[str]) -> None:
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    exists = path.exists()
    with path.open('a', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        if not exists:
            writer.writeheader()
        writer.writerows(rows)


def load_existing_sample_ids(path: Path) -> set[str]:
    if not path.exists():
        return set()
    with path.open(newline='', encoding='utf-8') as handle:
        return {row['sample_id'] for row in csv.DictReader(handle)}


def target_id(parts: list[str]) -> str:
    return hashlib.sha1('|'.join(parts).encode('utf-8')).hexdigest()[:16]


def sample_id_for(parts: list[str]) -> str:
    return 'inline_' + hashlib.sha1('|'.join(parts).encode('utf-8')).hexdigest()[:16]


def random_time_between(start: date, end: date, rng: random.Random) -> datetime:
    start_dt = datetime.combine(start, time.min, tzinfo=timezone.utc)
    end_dt = datetime.combine(end + timedelta(days=1), time.min, tzinfo=timezone.utc)
    seconds = int((end_dt - start_dt).total_seconds())
    return start_dt + timedelta(seconds=rng.randrange(max(1, seconds)))


def choose_background_center(region: str, bbox: tuple[float, float, float, float], hard_type: str, rng: random.Random) -> tuple[float, float]:
    west, south, east, north = bbox
    if region == 'south_america':
        zones = {
            'ocean_glint': (-50.0, -34.0, -35.0, 5.0), 'coastline': (-81.0, -35.0, -72.0, 10.0),
            'desert_hot_surface': (-72.0, -28.0, -66.0, -16.0), 'urban_industrial_heat': (-47.5, -24.5, -43.0, -20.0),
            'volcano_geothermal': (-73.0, -42.0, -66.0, -15.0), 'dust': (-70.0, -35.0, -58.0, -20.0),
            'fog_low_cloud': (-80.0, -40.0, -70.0, -10.0), 'agricultural_burn': (-64.0, -25.0, -50.0, -8.0),
            'deep_convective_cloud': (-75.0, -12.0, -48.0, 8.0), 'thin_cirrus': (-75.0, -30.0, -45.0, 10.0),
            'cloud_edge': (-78.0, -20.0, -45.0, 8.0), 'sensor_edge_artifact': (-82.0, -55.0, -76.0, 12.0),
        }
        zone = zones.get(hard_type)
        if zone:
            zw, zs, ze, zn = zone
            return rng.uniform(max(south, zs), min(north, zn)), rng.uniform(max(west, zw), min(east, ze))
    return rng.uniform(south, north), rng.uniform(west, east)


def build_targets(labels: list[dict[str, str]], cfg: RegionalConfig, rng: random.Random) -> list[dict[str, Any]]:
    start, end = date.fromisoformat(START_DATE), date.fromisoformat(END_DATE)
    positives = []
    for row in labels:
        label = int(row['label'])
        event_time = parse_utc(row['event_time_utc'])
        if label in FIRE_SIGNAL_LABELS and start <= event_time.date() <= end:
            positives.append(row)
    rng.shuffle(positives)
    positives = positives[:SAMPLES_PER_CLASS]
    targets = []
    for row in positives:
        label = int(row['label'])
        event_time = parse_utc(row['event_time_utc'])
        targets.append({'target_id': target_id([cfg.name, row['event_id'], row['event_time_utc'], str(label)]), 'target_kind': 'positive', 'label': label, 'label_name': LABEL_NAMES[label], 'event_id': row['event_id'], 'event_time_utc': row['event_time_utc'], 'target_time_utc': format_utc(event_time), 'center_lat': row['lat'], 'center_lon': row['lon'], 'label_source': row.get('source', 'regional_labels'), 'hard_negative_type': ''})
    hard_count = int(round(len(positives) * max(0.0, min(1.0, HARD_NEGATIVE_RATIO))))
    regular_count = len(positives) - hard_count
    for idx in range(regular_count):
        target_time = random_time_between(start, end, rng)
        lat, lon = choose_background_center(cfg.name, cfg.bbox, '', rng)
        targets.append({'target_id': target_id([cfg.name, 'negative', str(idx), format_utc(target_time), f'{lat:.5f}', f'{lon:.5f}']), 'target_kind': 'negative', 'label': 0, 'label_name': LABEL_NAMES[0], 'event_id': '', 'event_time_utc': '', 'target_time_utc': format_utc(target_time), 'center_lat': f'{lat:.6f}', 'center_lon': f'{lon:.6f}', 'label_source': 'regional_background_sampler', 'hard_negative_type': ''})
    for idx in range(hard_count):
        target_time = random_time_between(start, end, rng)
        hard_type = rng.choice(HARD_NEGATIVE_TYPES)
        lat, lon = choose_background_center(cfg.name, cfg.bbox, hard_type, rng)
        targets.append({'target_id': target_id([cfg.name, 'hard_negative', str(idx), hard_type, format_utc(target_time), f'{lat:.5f}', f'{lon:.5f}']), 'target_kind': 'hard_negative', 'label': 3, 'label_name': LABEL_NAMES[3], 'event_id': '', 'event_time_utc': '', 'target_time_utc': format_utc(target_time), 'center_lat': f'{lat:.6f}', 'center_lon': f'{lon:.6f}', 'label_source': 'regional_background_sampler', 'hard_negative_type': hard_type})
    rng.shuffle(targets)
    return targets


def goes_stamp_to_utc(key: str) -> datetime | None:
    match = GOES_START_RE.search(key)
    if not match:
        return None
    stamp = match.group('stamp')
    return datetime(int(stamp[0:4]), 1, 1, tzinfo=timezone.utc) + timedelta(days=int(stamp[4:7])-1, hours=int(stamp[7:9]), minutes=int(stamp[9:11]), seconds=int(stamp[11:13]), milliseconds=int(stamp[13]) * 100)


def hour_prefix(product: str, dt: datetime) -> str:
    return f'{product}/{dt.year}/{dt:%j}/{dt:%H}/'


def list_keys(s3: Any, bucket: str, prefix: str) -> list[str]:
    keys = []
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket=bucket, Prefix=prefix):
        for item in page.get('Contents', []):
            key = item['Key']
            if key.endswith('.nc'):
                keys.append(key)
    return sorted(keys)


def nearest_key_for_time(s3: Any, buckets: list[str], product: str, target: datetime) -> tuple[str, str, datetime] | None:
    ranked = []
    for bucket in buckets:
        for offset in (0, -1, 1):
            hour = target.replace(minute=0, second=0, microsecond=0) + timedelta(hours=offset)
            for key in list_keys(s3, bucket, hour_prefix(product, hour)):
                stamp = goes_stamp_to_utc(key)
                if stamp is not None:
                    ranked.append((abs((stamp - target).total_seconds()), bucket, key, stamp))
        if ranked:
            break
    if not ranked:
        return None
    _, bucket, key, stamp = sorted(ranked, key=lambda item: item[0])[0]
    return bucket, key, stamp


def nearest_sequence_for_time(s3: Any, buckets: list[str], product: str, anchor: datetime, offsets_minutes: tuple[int, ...]) -> tuple[tuple[str, str, datetime], ...] | None:
    seq = []
    for offset in offsets_minutes:
        match = nearest_key_for_time(s3, buckets, product, anchor + timedelta(minutes=int(offset)))
        if match is None:
            return None
        seq.append(match)
    return tuple(seq)


def download_one(s3: Any, bucket: str, key: str, out_path: Path) -> str:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and out_path.stat().st_size > 0:
        return 'skipped'
    tmp = out_path.with_suffix(out_path.suffix + '.part')
    if tmp.exists() and tmp.stat().st_size == 0:
        tmp.unlink()
    s3.download_file(bucket, key, str(tmp))
    tmp.replace(out_path)
    return 'downloaded'


def projection_transformer(ds: Any) -> tuple[Any, float]:
    from pyproj import CRS, Transformer
    proj = ds['goes_imager_projection']
    h = float(proj.attrs['perspective_point_height'])
    lon0 = float(proj.attrs['longitude_of_projection_origin'])
    sweep = str(proj.attrs.get('sweep_angle_axis', 'x'))
    a = float(proj.attrs['semi_major_axis'])
    b = float(proj.attrs['semi_minor_axis'])
    crs = CRS.from_proj4(f'+proj=geos +h={h} +lon_0={lon0} +sweep={sweep} +a={a} +b={b} +units=m +no_defs')
    return Transformer.from_crs('EPSG:4326', crs, always_xy=True), h


def latlon_to_rowcol(ds: Any, lat: float, lon: float, variable: str) -> tuple[int, int]:
    transformer, h = projection_transformer(ds)
    px, py = transformer.transform(float(lon), float(lat))
    da = ds[variable]
    y_name, x_name = da.dims[-2], da.dims[-1]
    xs = np.asarray(ds[x_name].values, dtype=np.float64)
    ys = np.asarray(ds[y_name].values, dtype=np.float64)
    col = int(np.abs(xs - (px / h)).argmin())
    row = int(np.abs(ys - (py / h)).argmin())
    return row, col


def crop_2d(arr: np.ndarray, center_row: int, center_col: int, size: int) -> tuple[np.ndarray, int, int]:
    half = int(size) // 2
    row0, col0 = int(center_row) - half, int(center_col) - half
    out = np.zeros((int(size), int(size)), dtype=np.float32)
    src_r0, src_c0 = max(0, row0), max(0, col0)
    src_r1, src_c1 = min(arr.shape[0], row0 + int(size)), min(arr.shape[1], col0 + int(size))
    dst_r0, dst_c0 = src_r0 - row0, src_c0 - col0
    if src_r1 > src_r0 and src_c1 > src_c0:
        out[dst_r0:dst_r0+(src_r1-src_r0), dst_c0:dst_c0+(src_c1-src_c0)] = arr[src_r0:src_r1, src_c0:src_c1]
    return out, row0, col0


def load_patch_from_open_dataset(ds: Any, lat: float, lon: float, channels: list[str], size: int) -> tuple[np.ndarray, int, int]:
    row, col = latlon_to_rowcol(ds, lat, lon, channels[0])
    bands, row0, col0 = [], 0, 0
    for channel in channels:
        if channel not in ds:
            raise KeyError(f'{channel} missing from dataset')
        crop, row0, col0 = crop_2d(np.asarray(ds[channel].values, dtype=np.float32), row, col, size)
        bands.append(crop)
    return np.stack(bands, axis=0), row0, col0


def write_sample(path: Path, x: np.ndarray, target: dict[str, Any], cfg: RegionalConfig, timestamp_utc: str, temporal_offsets: tuple[int, ...], temporal_timestamps: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, x=x.astype(np.float32, copy=False), class_label=np.asarray(int(target['label']), dtype=np.int64), center_lat=np.asarray(float(target['center_lat']), dtype=np.float32), center_lon=np.asarray(float(target['center_lon']), dtype=np.float32), timestamp_utc=np.asarray(timestamp_utc), temporal_offsets_minutes=np.asarray(temporal_offsets, dtype=np.int16), temporal_timestamps_utc=np.asarray(temporal_timestamps), source_id=np.asarray(cfg.goes_source), sector=np.asarray(cfg.sector), channel_names=np.asarray(list(cfg.raw_channels)), derived_channel_names=np.asarray([]))


def build_goes_samples(cfg: RegionalConfig, data_root: Path) -> Path:
    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config
    import xarray as xr
    from contextlib import ExitStack
    import time

    def log(message: str) -> None:
        print(f"[{datetime.now(timezone.utc).strftime('%H:%M:%S')} UTC] {message}", flush=True)

    t0 = time.perf_counter()
    rng = random.Random(SEED)
    region_root = cfg.region_root(data_root)
    labels_path = region_root / 'processed' / 'labels.csv'
    labels = read_csv(labels_path)
    if not labels:
        raise RuntimeError('No labels found. Run FIRMS download/ingest first.')

    log(f'start sample build region={cfg.name} labels={len(labels)} labels_path={labels_path}')
    log(f'config offsets={TEMPORAL_OFFSETS} samples_per_class={SAMPLES_PER_CLASS} max_goes_files={MAX_GOES_FILES} max_targets_per_file={MAX_TARGETS_PER_FILE} keep_raw={KEEP_RAW}')

    for folder in ['positive', 'early_positive', 'negative', 'hard_negative', 'uncertain']:
        (region_root / 'samples' / folder).mkdir(parents=True, exist_ok=True)
    manifest_path = region_root / 'manifest.csv'
    existing = load_existing_sample_ids(manifest_path)
    temporal_offsets = parse_temporal_offsets(TEMPORAL_OFFSETS)
    targets = build_targets(labels, cfg, rng)
    if not targets:
        raise RuntimeError('No targets selected.')

    target_counts = Counter(LABEL_NAMES[int(t['label'])] for t in targets)
    log(f'selected targets total={len(targets)} by_label={dict(target_counts)} existing_manifest_samples={len(existing)}')

    s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
    buckets = [cfg.bucket, *cfg.fallback_buckets]
    log(f'planning temporal sequences from buckets={buckets} product={cfg.product}')

    grouped: dict[tuple[tuple[str, str, str], ...], list[dict[str, Any]]] = defaultdict(list)
    missing = 0
    last_heartbeat = time.perf_counter()
    for idx, target in enumerate(targets, start=1):
        if idx == 1 or idx % 25 == 0:
            log(f'planning target {idx}/{len(targets)} grouped_sequences={len(grouped)} missing={missing}')
        seq = nearest_sequence_for_time(s3, buckets, cfg.product, parse_utc(str(target['target_time_utc'])), temporal_offsets)
        if seq is None:
            missing += 1
            continue
        target['temporal_sequence'] = [(bucket, key, format_utc(stamp)) for bucket, key, stamp in seq]
        target['timestamp_utc'] = format_utc(seq[-1][2])
        grouped[tuple(target['temporal_sequence'])].append(target)
        now = time.perf_counter()
        if now - last_heartbeat > 30:
            log(f'heartbeat planning idx={idx}/{len(targets)} grouped_sequences={len(grouped)} missing={missing}')
            last_heartbeat = now

    file_items = list(grouped.items())
    planned_sequences = len(file_items)
    if MAX_GOES_FILES > 0:
        file_items = file_items[:MAX_GOES_FILES]
    log(f'planning complete targets={len(targets)} missing={missing} planned_sequences={planned_sequences} selected_sequences={len(file_items)} elapsed={time.perf_counter()-t0:.1f}s')

    if not file_items:
        raise RuntimeError('No GOES temporal sequences found. Try wider TEMPORAL_OFFSETS like -60,-40,-20,0 or a different date range.')

    counts = defaultdict(int)
    for file_idx, (sequence, file_targets) in enumerate(file_items, start=1):
        seq_t0 = time.perf_counter()
        original_target_count = len(file_targets)
        if MAX_TARGETS_PER_FILE > 0:
            file_targets = file_targets[:MAX_TARGETS_PER_FILE]
        log(f'sequence {file_idx}/{len(file_items)} start original_targets={original_target_count} processing_targets={len(file_targets)} files={len(sequence)}')
        for seq_i, (bucket, key, ts) in enumerate(sequence, start=1):
            log(f'  file {seq_i}/{len(sequence)} ts={ts} s3://{bucket}/{key}')

        sequence_paths = []
        for seq_i, (bucket, key, ts) in enumerate(sequence, start=1):
            local_path = region_root / 'raw' / 'streaming' / bucket / key
            d0 = time.perf_counter()
            status = download_one(s3, bucket, key, local_path)
            dt = time.perf_counter() - d0
            counts[status] += 1
            size_mb = local_path.stat().st_size / 1_000_000 if local_path.exists() else 0.0
            log(f'  download {seq_i}/{len(sequence)} status={status} size_mb={size_mb:.1f} elapsed={dt:.1f}s')
            sequence_paths.append(local_path)

        rows = []
        try:
            open_t0 = time.perf_counter()
            with ExitStack() as stack:
                datasets = [stack.enter_context(xr.open_dataset(path, engine='netcdf4')) for path in sequence_paths]
                log(f'  opened {len(datasets)} NetCDF files elapsed={time.perf_counter()-open_t0:.1f}s')
                for target_idx, target in enumerate(file_targets, start=1):
                    if target_idx == 1 or target_idx % 25 == 0 or target_idx == len(file_targets):
                        log(f'  extracting target {target_idx}/{len(file_targets)} samples_written_total={counts["samples_written"]} failed_total={counts["sample_failed"]}')
                    label = int(target['label'])
                    timestamp_utc = str(target['timestamp_utc'])
                    temporal_timestamps = [str(row[2]) for row in target['temporal_sequence']]
                    sid = sample_id_for([cfg.name, target['target_id'], '|'.join(temporal_timestamps), str(label)])
                    if sid in existing:
                        counts['sample_skipped'] += 1
                        continue
                    try:
                        frames, row0, col0 = [], 0, 0
                        for ds in datasets:
                            frame, row0, col0 = load_patch_from_open_dataset(ds, float(target['center_lat']), float(target['center_lon']), list(cfg.raw_channels), cfg.patch_size)
                            frames.append(frame)
                        x = np.stack(frames, axis=0)
                    except Exception as exc:
                        counts['sample_failed'] += 1
                        log(f'  failed sample target_id={target["target_id"]} label={LABEL_NAMES[label]} error={exc}')
                        continue
                    out_path = region_root / 'samples' / folder_for_label(label) / f'{sid}.npz'
                    write_sample(out_path, x, target, cfg, timestamp_utc, temporal_offsets, temporal_timestamps)
                    existing.add(sid)
                    target_time = parse_utc(target.get('event_time_utc') or target['target_time_utc'])
                    delta = (parse_utc(timestamp_utc) - target_time).total_seconds() / 60.0
                    rows.append({'sample_id': sid, 'region': cfg.name, 'source_id': cfg.goes_source, 'sector': cfg.sector, 'timestamp_utc': timestamp_utc, 'center_lat': target['center_lat'], 'center_lon': target['center_lon'], 'bbox_w': cfg.bbox[0], 'bbox_s': cfg.bbox[1], 'bbox_e': cfg.bbox[2], 'bbox_n': cfg.bbox[3], 'patch_row0': row0, 'patch_col0': col0, 'patch_size': cfg.patch_size, 'label': label, 'label_name': LABEL_NAMES[label], 'label_source': target['label_source'], 'event_id': target.get('event_id', ''), 'event_time_utc': target.get('event_time_utc', ''), 'delta_t_minutes': f'{delta:.2f}', 'hard_negative_type': target.get('hard_negative_type', ''), 'split': split_for_id(sid), 'path': str(out_path.relative_to(region_root))})
                    counts['samples_written'] += 1
        finally:
            if not KEEP_RAW:
                for path in set(sequence_paths):
                    path.unlink(missing_ok=True)
        append_csv(manifest_path, rows, MANIFEST_COLUMNS)
        log(f'sequence {file_idx}/{len(file_items)} done rows_appended={len(rows)} elapsed={time.perf_counter()-seq_t0:.1f}s counts={dict(counts)}')

    log(f'sample build done manifest={manifest_path} elapsed={time.perf_counter()-t0:.1f}s')
    print(json.dumps({'manifest': str(manifest_path), 'targets': len(targets), 'missing_targets': missing, 'selected_sequences': len(file_items), 'counts': dict(counts)}, indent=2), flush=True)
    return manifest_path



In [ ]:
# Dataset, training, evaluation, and report helpers
CLASS_LABEL_COLUMNS = ('class_label', 'y_class', 'label')


def normalize_patch(x: np.ndarray) -> np.ndarray:
    if x.ndim == 4:
        return np.stack([normalize_patch(frame) for frame in x], axis=0)
    x = x.astype(np.float32, copy=False)
    out = np.zeros_like(x, dtype=np.float32)
    for idx in range(x.shape[0]):
        band = x[idx]
        valid = band[np.isfinite(band)]
        if valid.size == 0:
            continue
        lo, hi = np.percentile(valid, [1, 99])
        if hi > lo:
            out[idx] = np.clip((band - lo) / (hi - lo), 0.0, 1.0)
    return np.nan_to_num(out, nan=0.0, posinf=1.0, neginf=0.0)


class RegionalGoesFireDataset(Dataset):
    def __init__(self, manifest: Path, split: str, temporal_steps: int, uncertain_weight: float = 0.0):
        self.manifest = Path(manifest)
        self.root = self.manifest.parent
        with self.manifest.open(newline='', encoding='utf-8') as handle:
            rows = [row for row in csv.DictReader(handle) if row.get('split') == split]
        self.rows = [row for row in rows if self.sample_path(row).exists()]
        self.temporal_steps = int(temporal_steps)
        self.uncertain_weight = float(uncertain_weight)
        if not self.rows:
            raise RuntimeError(f'No readable samples for split={split} in {manifest}')

    def sample_path(self, row: dict[str, str]) -> Path:
        p = Path(row['path'])
        return p if p.is_absolute() else self.root / p

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        row = self.rows[int(idx)]
        path = self.sample_path(row)
        with np.load(path, allow_pickle=False) as npz:
            x = normalize_patch(npz['x'].astype(np.float32, copy=False))
            if x.ndim != 4 or x.shape[0] != self.temporal_steps:
                raise ValueError(f'Expected [{self.temporal_steps},C,H,W], got {x.shape} in {path}')
            label = int(np.asarray(npz['class_label']).reshape(-1)[0]) if 'class_label' in npz else int(row['label'])
        return {
            'image': torch.from_numpy(x),
            'class_label': torch.tensor(binary_label_for(label), dtype=torch.long),
            'original_class_label': torch.tensor(label, dtype=torch.long),
            'sample_weight': torch.tensor(self.uncertain_weight if label in UNCERTAIN_LABELS else 1.0, dtype=torch.float32),
            'sample_id': row.get('sample_id', path.stem),
            'hard_negative_type': row.get('hard_negative_type', ''),
            'delta_t_minutes': row.get('delta_t_minutes', ''),
        }


def collate(batch):
    return {
        'image': torch.nan_to_num(torch.stack([b['image'] for b in batch]).float(), nan=0.0, posinf=1.0, neginf=0.0),
        'class_label': torch.stack([b['class_label'] for b in batch]).long(),
        'original_class_label': torch.stack([b['original_class_label'] for b in batch]).long(),
        'sample_weight': torch.stack([b['sample_weight'] for b in batch]).float(),
        'sample_id': [b['sample_id'] for b in batch],
        'hard_negative_type': [b['hard_negative_type'] for b in batch],
        'delta_t_minutes': [b['delta_t_minutes'] for b in batch],
    }


def weighted_bce(logits: torch.Tensor, labels: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    logits = logits.reshape(-1)
    targets = labels.to(dtype=logits.dtype).reshape(-1)
    weights = weights.to(dtype=logits.dtype).reshape(-1)
    per_sample = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    return (per_sample * weights).sum() / weights.sum().clamp_min(1.0)


@torch.no_grad()
def evaluate_loader(model: nn.Module, loader: DataLoader, device: torch.device) -> dict[str, float]:
    model.eval()
    total_loss = total = correct = 0
    tp = fp = fn = 0
    for batch in loader:
        x, y, w = batch['image'].to(device), batch['class_label'].to(device), batch['sample_weight'].to(device)
        logits = model(x)['scene_logits']
        loss = weighted_bce(logits, y, w)
        pred = torch.sigmoid(logits.reshape(-1)) >= THRESHOLD
        valid = w > 0
        correct += int((pred[valid] == y[valid]).sum().item())
        total += int(valid.sum().item())
        target = y.bool()
        tp += int((pred & target & valid).sum().item())
        fp += int((pred & ~target & valid).sum().item())
        fn += int((~pred & target & valid).sum().item())
        total_loss += float(loss.item()) * int(x.shape[0])
    return {'loss': total_loss / max(1, len(loader.dataset)), 'accuracy': correct / max(1, total), 'precision': tp / max(1, tp + fp), 'recall': tp / max(1, tp + fn), 'f1': (2 * tp) / max(1, 2 * tp + fp + fn)}


def pick_torch_device() -> torch.device:
    if not torch.cuda.is_available():
        print('CUDA not available; training on CPU')
        return torch.device('cpu')
    name = torch.cuda.get_device_name(0)
    capability = torch.cuda.get_device_capability(0)
    arches = torch.cuda.get_arch_list()
    required_arch = f'sm_{capability[0]}{capability[1]}'
    print(f'CUDA device: {name} capability={capability} required_arch={required_arch}')
    print(f'PyTorch compiled arches: {arches}')
    if required_arch not in arches:
        print(f'WARNING: this PyTorch build does not include {required_arch}; falling back to CPU')
        return torch.device('cpu')
    try:
        probe = torch.nn.Conv2d(1, 1, 1).cuda()
        _ = probe(torch.zeros(1, 1, 4, 4, device='cuda'))
        torch.cuda.synchronize()
    except Exception as exc:
        print(f'WARNING: CUDA probe failed ({type(exc).__name__}: {exc}); falling back to CPU')
        return torch.device('cpu')
    return torch.device('cuda')

def train_model(manifest: Path, cfg: RegionalConfig, data_root: Path) -> Path:
    temporal_steps = len(parse_temporal_offsets(TEMPORAL_OFFSETS))
    train_ds = RegionalGoesFireDataset(manifest, 'train', temporal_steps)
    val_ds = RegionalGoesFireDataset(manifest, 'val', temporal_steps)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate)
    device = pick_torch_device()
    model = SentinelaModel(SentinelaConfig(in_channels=len(cfg.raw_channels) * temporal_steps, mask_classes=1, scene_classes=1, temporal_steps=temporal_steps, input_channels_per_timestep=len(cfg.raw_channels), variant=VARIANT, input_size=cfg.patch_size)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    out_dir = Path(WORK_ROOT) / 'models' / 'regional' / cfg.name
    out_dir.mkdir(parents=True, exist_ok=True)
    history, best_score = [], -1.0
    print(f'device={device} train={len(train_ds)} val={len(val_ds)} out_dir={out_dir}', flush=True)
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = total = 0
        for batch in train_loader:
            x, y, w = batch['image'].to(device), batch['class_label'].to(device), batch['sample_weight'].to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = weighted_bce(model(x)['scene_logits'], y, w)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * int(x.shape[0])
            total += int(x.shape[0])
        val = evaluate_loader(model, val_loader, device)
        row = {'epoch': epoch, 'train_loss': total_loss / max(1, total), 'val_loss': val['loss'], 'val_accuracy': val['accuracy'], 'val_precision': val['precision'], 'val_recall': val['recall'], 'val_f1': val['f1']}
        history.append(row)
        print(json.dumps(row, sort_keys=True), flush=True)
        checkpoint = {'model': model.state_dict(), 'model_config': model.config.__dict__, 'channels': list(cfg.raw_channels), 'temporal_offsets_minutes': parse_temporal_offsets(TEMPORAL_OFFSETS), 'label_contract': {'0': 'no_fire', '1': 'fire_signal'}, 'history': history, 'epoch': epoch, 'metrics': row}
        torch.save(checkpoint, out_dir / 'latest.pt')
        score = 0.5 * row['val_precision'] + 0.5 * row['val_recall']
        if score > best_score:
            best_score = score
            torch.save(checkpoint, out_dir / 'best.pt')
    (out_dir / 'history.json').write_text(json.dumps(history, indent=2) + '\n')
    return out_dir / 'best.pt'


def as_float(value: Any) -> float | None:
    try:
        return None if value in ('', None) else float(value)
    except Exception:
        return None


@torch.no_grad()
def evaluate_checkpoint(manifest: Path, checkpoint_path: Path, cfg: RegionalConfig, data_root: Path) -> Path:
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model = SentinelaModel(SentinelaConfig(**checkpoint['model_config']))
    model.load_state_dict(checkpoint['model'])
    device = pick_torch_device()
    model.to(device).eval()
    temporal_steps = len(tuple(checkpoint.get('temporal_offsets_minutes', parse_temporal_offsets(TEMPORAL_OFFSETS))))
    test_ds = RegionalGoesFireDataset(manifest, 'test', temporal_steps)
    loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate)
    confusion = [[0, 0], [0, 0]]
    original_counts = Counter()
    hard_fp = Counter()
    lead = defaultdict(list)
    tp = fp = fn = correct = total = 0
    for batch in loader:
        x = batch['image'].to(device)
        y = batch['class_label'].to(device)
        logits = model(x)['scene_logits'].reshape(-1)
        pred = (torch.sigmoid(logits) >= THRESHOLD).long().cpu()
        for idx, (target, predicted, original) in enumerate(zip(y.cpu().tolist(), pred.tolist(), batch['original_class_label'].cpu().tolist(), strict=True)):
            original_counts[LABEL_NAMES[int(original)]] += 1
            confusion[int(target)][int(predicted)] += 1
            correct += int(predicted == target); total += 1
            pred_fire, target_fire = predicted == 1, target == 1
            tp += int(pred_fire and target_fire); fp += int(pred_fire and not target_fire); fn += int((not pred_fire) and target_fire)
            if original == 3 and pred_fire:
                hard_fp[batch['hard_negative_type'][idx] or 'unknown'] += 1
            delta = as_float(batch['delta_t_minutes'][idx])
            if delta is not None:
                lead[LABEL_NAMES[int(original)]].append(delta)
    result = {'region': cfg.name, 'checkpoint': str(checkpoint_path), 'split': 'test', 'threshold': THRESHOLD, 'temporal_offsets_minutes': list(parse_temporal_offsets(TEMPORAL_OFFSETS)), 'accuracy': correct / max(1, total), 'precision': tp / max(1, tp + fp), 'recall': tp / max(1, tp + fn), 'f1': (2 * tp) / max(1, 2 * tp + fp + fn), 'confusion_matrix': confusion, 'labels': BINARY_LABEL_NAMES, 'original_labels': LABEL_NAMES, 'original_class_counts': dict(original_counts), 'false_positives_by_hard_negative_type': dict(hard_fp), 'delta_t_minutes': {name: {'count': len(values), 'mean': sum(values) / max(1, len(values))} for name, values in lead.items()}}
    out_json = cfg.region_root(data_root) / 'evaluation' / 'best_test.json'
    out_json.parent.mkdir(parents=True, exist_ok=True)
    out_json.write_text(json.dumps(result, indent=2) + '\n')
    print(json.dumps(result, indent=2), flush=True)
    report = cfg.region_root(data_root) / 'reports' / 'regional_benchmark.md'
    report.parent.mkdir(parents=True, exist_ok=True)
    report.write_text(f"# Regional Benchmark: {cfg.name}\n\n- Precision: {result['precision']:.4f}\n- Recall: {result['recall']:.4f}\n- F1: {result['f1']:.4f}\n- Labels: `{json.dumps(BINARY_LABEL_NAMES)}`\n- Original counts: `{json.dumps(dict(original_counts), sort_keys=True)}`\n", encoding='utf-8')
    return out_json

## Run Pipeline

In [ ]:
# 1. Get labels
cfg = get_region_config(REGION)
raw_labels = download_firms_labels(cfg, DATA_ROOT)
processed_labels = ingest_labels(raw_labels, cfg, DATA_ROOT)

In [ ]:
# 2. Build temporal GOES samples
manifest = build_goes_samples(cfg, DATA_ROOT)

In [ ]:
# 3. Inspect manifest and one sample
rows = read_csv(manifest)
print('manifest', manifest, 'rows', len(rows))
print('labels', Counter(row.get('label_name', '') for row in rows))
print('splits', Counter(row.get('split', '') for row in rows))
if rows:
    first = cfg.region_root(DATA_ROOT) / rows[0]['path']
    with np.load(first, allow_pickle=False) as sample:
        print('first_sample', first)
        print('x_shape', sample['x'].shape)
        print('class_label', int(sample['class_label']))
        print('temporal_offsets', sample['temporal_offsets_minutes'].tolist())

In [ ]:
# 4. Train
best_checkpoint = train_model(manifest, cfg, DATA_ROOT)
print('best_checkpoint', best_checkpoint)

In [ ]:
# 5. Evaluate and write report
eval_json = evaluate_checkpoint(manifest, best_checkpoint, cfg, DATA_ROOT)
print('eval_json', eval_json)

In [ ]:
# 6. Final artifact paths
artifacts = {
    'best_checkpoint': str(best_checkpoint),
    'history': str(best_checkpoint.parent / 'history.json'),
    'evaluation': str(eval_json),
    'report': str(cfg.region_root(DATA_ROOT) / 'reports' / 'regional_benchmark.md'),
    'manifest': str(manifest),
}
print(json.dumps(artifacts, indent=2))

## Real Run Settings

After a smoke run succeeds:

```python
MAX_FIRMS_ROWS = 0
SAMPLES_PER_CLASS = 10000
MAX_GOES_FILES = 0
VARIANT = 's'  # or 'm'
EPOCHS = 20
```

If GOES availability is sparse, use:

```python
TEMPORAL_OFFSETS = '-60,-40,-20,0'
```